# CloudGuard AI — AWS Permission Review Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gopika252006/CloudGuard/blob/main/cloudguard_ai_demo.ipynb)

**Catch over-permissioned AWS IAM requests before they reach production.**

This notebook walks through the full CloudGuard AI pipeline:

| Step | What happens |
|---|---|
| 1 | Extract the IAM action and resource ARN from a plain-English request |
| 2 | Deterministic safety checks — wildcards, admin, vague requests blocked instantly |
| 3 | Strands Decider 2B classifies the request with 4 typed yes/no questions |
| 4 | Return `allow`, `clarify`, or `human_review` with a confidence score |

**Tags:** `#strands-decider-2b` `#policy`  
**Repo:** https://github.com/Gopika252006/CloudGuard  
**Deployed on:** Amazon Bedrock AgentCore Runtime (`us-east-1`)

> All permission recommendations are simulated. CloudGuard AI never creates, modifies, or applies real AWS IAM permissions.

---
*Built with the AWS User Group Madurai Strands Decider hackathon kit*

## Setup
Clone the repo and install dependencies. Skip if running locally inside the project folder.

In [ ]:
import sys, os
from pathlib import Path

# If running in Colab, clone the repo first:
# !git clone https://github.com/Gopika252006/CloudGuard.git
# %cd CloudGuard

ROOT = Path('.').resolve()
for p in [str(ROOT), str(ROOT / 'tests')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Install runtime dependencies
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'strands-agents[openai]', 'fastapi', 'uvicorn', 'pyyaml'], check=True)

# Load .env configuration
from decider_kit.config import load_dotenv
load_dotenv()

print('LLM_PROVIDER :', os.environ.get('LLM_PROVIDER', 'bedrock'))
print('DECIDER_MODE :', os.environ.get('DECIDER_MODE', 'local'))
print('Ready.')

---
## 1. Resource Name Extraction

The first thing CloudGuard AI does is extract the actual resource name from the developer's natural-language request.

This was the root cause of the original bug — `arn:aws:s3:::lambda/*` was generated instead of `arn:aws:s3:::reports-bucket/*` because the regex matched the word "Lambda" (a service keyword) instead of "reports-bucket" (the actual bucket name).

The fix adds an optional service abbreviation pattern between the resource name and the type word, handling inputs like `"the reports-bucket S3 bucket"` and `"the orders DynamoDB table"`.

In [1]:
from agent.tools import _extract_resource_name

cases = [
    'My Lambda function needs to upload reports to the reports-bucket S3 bucket only.',
    'Allow the scheduled job to write the nightly CSV to the finance-reports S3 bucket.',
    'Read items from the orders DynamoDB table.',
    'Give me permission to access the data.',   # vague -> None
]

for r in cases:
    res = _extract_resource_name(r)
    print(f'{str(res):<22} <- {r[:55]}')

reports-bucket         <- My Lambda function needs to upload reports to the repor
finance-reports        <- Allow the scheduled job to write the nightly CSV to the
orders                 <- Read items from the orders DynamoDB table.
None                   <- Give me permission to access the data.


---
## 2. Deterministic Safety Checks

These rules run **before** Strands Decider 2B is called. They block obvious violations instantly with zero model cost:
- Wildcard actions (`s3:*`, `iam:*`, bare `*`)
- Admin access (`AdministratorAccess`, `PowerUserAccess`, `root`)
- Wildcard resources (`on *`)
- Vague requests with no service, action, or resource

In [2]:
from agent.decision import deterministic_check, is_vague_request

checks = [
    ('s3:PutObject on arn:aws:s3:::reports-bucket/*',
     'My Lambda needs to upload to the reports-bucket.'),
    ('AdministratorAccess on *',
     'Give this Lambda administrator access.'),
    ('s3:* on *',
     'Allow s3:* on every S3 resource.'),
    ('unknown action on unknown resource',
     'Give me permission to access the data.'),
]

for action, req in checks:
    det   = deterministic_check(action, req)
    vague = is_vague_request(req, action)
    block = det[0] if det else ('clarify(vague)' if vague else 'pass')
    print(f'{action[:48]:<50} {block}')

s3:PutObject on arn:aws:s3:::reports-bucket/*      pass
AdministratorAccess on *                           human_review
s3:* on *                                          human_review
unknown action on unknown resource                 clarify(vague)


---
## 3. The Four Strands Decider 2B Questions

For requests that pass the deterministic checks, CloudGuard AI sends these four typed yes/no questions to Strands Decider 2B in a **single API call**.

In [3]:
from agent.decision import QUESTIONS

for name, q in QUESTIONS.items():
    print(f'--- {name} ({q["type"]}) ---')
    print(f'  Q: {q["instructions"][:95]}')
    for opt, desc in (q.get('criteria') or {}).items():
        print(f'  {opt}:  {str(desc)[:90]}')
    print()

--- scope_appropriate (noul) ---
  Q: Is the proposed IAM resource a specific, named resource (not a wildcard or account-wide scope)?
  true:  the resource is an explicit named ARN or resource name: e.g. arn:aws:s3:::reports-bucket
  false: the resource is '*' (all resources), 'arn:aws:*:*:*:*', an account-wide wildcard

--- action_matches_task (noul) ---
  Q: Does the proposed IAM action list contain only the specific permissions the stated task requires?
  true:  every action in the list is needed for the task -- for example s3:PutObject for 'upload
  false: the action list is broader than the task needs -- for example s3:* or dynamodb:* when on

--- contains_prohibited (noul) ---
  Q: Does the proposed action include a wildcard service action (like s3:*, dynamodb:*, iam:*, or
  true:  yes -- the action is a service wildcard (s3:*, ec2:*, dynamodb:*, lambda:*, iam:*, bedro
  false: no -- the action is a list of specific named operations such as s3:PutObject, dynamodb:G

--- missing_i

---
## 4. Permission Review — Single Request

The canonical example from the bug report. The original code generated `arn:aws:s3:::lambda/*`. After the fix it correctly produces `arn:aws:s3:::reports-bucket/*`.

We use `FakeDecider` (returns 0.9 for all yes/no questions) so this cell runs offline without any API key. Replace with `Decider()` for a real review.

In [4]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('tests').resolve()))

from fakes import FakeDecider
from agent.decision import PermissionReviewer

# Swap FakeDecider() for Decider() when your UGMDU key is in .env
# from decider_kit import Decider
# reviewer = PermissionReviewer(Decider())
reviewer = PermissionReviewer(FakeDecider())

request  = 'My Lambda function needs to upload reports to the reports-bucket S3 bucket only.'
proposed = 's3:PutObject on arn:aws:s3:::reports-bucket/*'

result = reviewer.review(request, proposed)

print('Request  :', request)
print('Proposed :', proposed)
print()
print(f'Outcome     : {result.outcome}')
print(f'Confidence  : {result.confidence:.0%}')
print(f'Reason      : {result.reason}')
print(f'Recommend   : {result.recommendation[:90]}')
print(f'Decider ms  : {result.decider_ms}')
if result.answers:
    print()
    print('Decider answers:')
    for k, v in result.answers.items():
        p = v.get('p_yes') or 0
        bar = chr(9608) * int(p * 20)
        print(f'  {k:<26} p_yes={p:<6}  {bar}')

Request  : My Lambda function needs to upload reports to the reports-bucket S3 bucket only.
Proposed : s3:PutObject on arn:aws:s3:::reports-bucket/*

Outcome     : allow
Confidence  : 90%
Reason      : The request is specific, appropriately scoped, and matches the stated task.
Recommend   : Recommended: s3:PutObject on arn:aws:s3:::reports-bucket/* - least-privilege permission.
Decider ms  : 12

Decider answers:
  scope_appropriate         p_yes=0.9   ██████████████████
  action_matches_task       p_yes=0.9   ██████████████████
  contains_prohibited       p_yes=0.9   ██████████████████
  missing_info              p_yes=0.9   ██████████████████


---
## 5. All Demo Scenarios

These are the exact requests used in the CloudGuard AI demo video, showing all three outcome types.

In [5]:
from agent.agent import _build_proposed_action

scenarios = [
    'My Lambda function needs to upload reports to the reports-bucket S3 bucket only.',
    'Give this Lambda administrator access to every AWS service.',
    'Give me permission to access the data.',
    'Allow the scheduled job to write the nightly CSV report to the finance-reports bucket.',
    'Allow s3:* on every S3 resource.',
]

rev = PermissionReviewer(FakeDecider())

print(f'{"Outcome":<16} {"How decided":<24} Request')
print('-' * 80)
for req in scenarios:
    pa, _ = _build_proposed_action(req)
    r = rev.review(req, pa)
    how = 'deterministic' if r.decider_ms == 0 else f'Decider {r.confidence:.0%}'
    print(f'{r.outcome:<16} {how:<24} {req[:50]}...')

Outcome         How decided             Request
--------------------------------------------------------------------------------
allow           Decider 90%            My Lambda function needs to upload reports to the...
human_review    deterministic          Give this Lambda administrator access to every AW...
human_review    deterministic          Give me permission to access the data.
allow           Decider 90%            Allow the scheduled job to write the nightly CSV ...
human_review    deterministic          Allow s3:* on every S3 resource.


---
## 6. Outcome Logic

The deterministic layer runs first and cannot be overridden by the Decider.

In [6]:
diagram = """\
Developer request (natural language)
        |
        v
Amazon Bedrock (Nova Pro)
  interprets request -> proposes IAM action + resource ARN
        |
        v
Layer 1: Deterministic safety check  (no model cost)
  wildcard action / admin access / bare * resource -> human_review
  vague / no service / no action                  -> clarify
        |
        v  (specific, scoped requests only)
Layer 2: Strands Decider 2B  (4 questions, 1 API call)
  scope_appropriate    -- named resource, not wildcard?
  action_matches_task  -- minimal action for the task?
  contains_prohibited  -- admin / wildcard / full-service?
  missing_info         -- enough info to review safely?
        |
        v
Layer 3: Outcome logic  (fail-closed)
  missing_info = yes                         -> clarify
  contains_prohibited = yes (confident)      -> human_review
  scope bad AND action bad                   -> human_review
  scope bad OR action bad                    -> clarify
  all pass, confidence >= 65%               -> allow
  low confidence or Decider error            -> human_review"""

print(diagram)

Developer request (natural language)
        |
        v
Amazon Bedrock (Nova Pro)
  interprets request -> proposes IAM action + resource ARN
        |
        v
Layer 1: Deterministic safety check  (no model cost)
  wildcard action / admin access / bare * resource -> human_review
  vague / no service / no action                  -> clarify
        |
        v  (specific, scoped requests only)
Layer 2: Strands Decider 2B  (4 questions, 1 API call)
  scope_appropriate    -- named resource, not wildcard?
  action_matches_task  -- minimal action for the task?
  contains_prohibited  -- admin / wildcard / full-service?
  missing_info         -- enough info to review safely?
        |
        v
Layer 3: Outcome logic  (fail-closed)
  missing_info = yes                         -> clarify
  contains_prohibited = yes (confident)      -> human_review
  scope bad AND action bad                   -> human_review
  scope bad OR action bad                    -> clarify
  all pass, confidence >= 65% 

---
## 7. Benchmark Results

From `./kit bench --split test` — 14 held-out cases, 53 individual question answers.

Decider: `strands-decider-2b-hobson-v19` on the UGMDU hosted API.  
LLM baseline: `qwen3.5-4b` on the same hosted API.

In [7]:
import json
from pathlib import Path

bench_path = Path('submission/bench/results.json')
if not bench_path.exists():
    print('Run ./kit bench --split test to generate results.')
else:
    data = json.loads(bench_path.read_text())
    s    = data['summary']
    d, l = s['decider'], s['llm']

    rows = [
        ('Accuracy',                    f"{d['accuracy']:.1%}",                f"{l['accuracy']:.1%}"),
        ('Latency p50 (ms)',             f"{d['latency_p50_ms']:,.0f}",          f"{l['latency_p50_ms']:,.0f}"),
        ('Latency p95 (ms)',             f"{d['latency_p95_ms']:,.0f}",          f"{l['latency_p95_ms']:,.0f}"),
        ('Cost per 1,000 decisions ($)', f"${d['usd_per_1k_decisions']:.4f}",   f"${l['usd_per_1k_decisions']:.4f}"),
        ('Answered alone  (conf >= 0.7)',f"{d['confident_share']:.1%}",          f"{l['confident_share']:.1%}"),
        ('Accuracy when confident',      f"{d['accuracy_when_confident']:.1%}",  f"{l['accuracy_when_confident']:.1%}"),
    ]

    print(f"{'Metric':<36} {'Strands Decider 2B':>22} {'LLM Baseline':>14}")
    print('-' * 72)
    for label, dval, lval in rows:
        print(f"{label:<36} {dval:>22} {lval:>14}")

    print(f"\nSpeedup at p50: {s['speedup_p50']:.1f}x faster than the LLM")
    print()
    print('Note: the LLM is more accurate overall (larger model). The Decider\'s value is')
    print('calibrated confidence: 57% of cases get a fast, cheap answer at 86.7% accuracy;')
    print('the remaining 43% are escalated to human_review instead of guessed.')

Metric                              Strands Decider 2B      LLM Baseline
------------------------------------------------------------------------
Accuracy                                         75.5%         98.1%
Latency p50 (ms)                                 5,539        22,759
Latency p95 (ms)                                 6,054        23,145
Cost per 1,000 decisions ($)                    $0.0350        $0.0000
Answered alone  (conf >= 0.7)                    57.0%        100.0%
Accuracy when confident                          86.7%         98.1%

Speedup at p50: 4.1x faster than the LLM

Note: the LLM is more accurate overall (larger model). The Decider's value is
calibrated confidence: 57% of cases get a fast, cheap answer at 86.7% accuracy;
the remaining 43% are escalated to human_review instead of guessed.


---
## 8. Full End-to-End Review (requires credentials)

This runs the complete `HackathonAgent.chat()` pipeline with the real LLM and Strands Decider 2B.

**Requirements:** Set `LLM_PROVIDER` and `DECIDER_MODE` in `.env` with valid API keys or AWS credentials.

In [ ]:
# Uncomment to run with real credentials:
#
# from agent.agent import HackathonAgent
#
# agent = HackathonAgent()
# out   = agent.chat(
#     'My Lambda function needs to upload reports to the reports-bucket S3 bucket only.'
# )
#
# print('Reply:', out['reply'][:300])
# if out['decisions']:
#     d = out['decisions'][0]
#     print(f"Outcome    : {d['outcome']}")
#     print(f"Confidence : {d['confidence']:.0%}")
#     print(f"Proposed   : {d['proposed_action']}")
# print(f"Latency    : {out['latency_ms']} ms total")

print('Cell skipped — add credentials and uncomment to run the full pipeline.')

---
## Summary

| Component | Technology |
|---|---|
| Agent framework | Strands Agents SDK |
| LLM | Amazon Bedrock (Nova Pro) |
| Decision model | Strands Decider 2B |
| Deployment | Amazon Bedrock AgentCore Runtime |
| Pattern | `#policy` — IAM permission scope review |

**Key design principles:**
- Deterministic rules catch obvious violations at zero model cost
- Strands Decider 2B handles nuanced language understanding with calibrated confidence
- Fail-closed: uncertainty always escalates to `human_review`, never approves
- All output is a recommendation — no real AWS permissions are ever modified

---
**Repo:** https://github.com/Gopika252006/CloudGuard  
**Tags:** `#strands-decider-2b` `#policy`  
*Built with the AWS User Group Madurai Strands Decider hackathon kit*